In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import itertools

spark = SparkSession.builder \
    .appName("TMDB Exploratory Data Analysis") \
    .getOrCreate()

In [58]:
movies_df = spark.read.parquet("../data/processed/tmdb_movies_clean.parquet")
movies_df.cache()  # Cache karena akan dipakai beberapa kali
movies_df.show(5, truncate=False)

+------------------+----+-----------+--------+----------------------------------+-----------+-----+--------------+-------------------------------------------------------------+------------+-----------------+----------+-------+
|Movie Name        |Year|Certificate|Duration|Genre                             |TMDB Rating|Votes|Director      |Stars                                                        |Grossed in $|Original Language|Popularity|TMDB ID|
+------------------+----+-----------+--------+----------------------------------+-----------+-----+--------------+-------------------------------------------------------------+------------+-----------------+----------+-------+
|American Beauty   |1999|R          |122     |Drama                             |8.001      |12912|Sam Mendes    |[Kevin Spacey, Annette Bening, Thora Birch, Wes Bentley]     |3.56296601E8|en               |7.7545    |14     |
|Citizen Kane      |1941|NR         |119     |Mystery, Drama                    |7.978      

## Statistika Deskriptif

In [59]:
movies_df.select(
    "Year", "Duration", "TMDB Rating", "Votes", "Grossed in $", "Popularity"
).describe().show()

+-------+------------------+-----------------+------------------+------------------+--------------------+-----------------+
|summary|              Year|         Duration|       TMDB Rating|             Votes|        Grossed in $|       Popularity|
+-------+------------------+-----------------+------------------+------------------+--------------------+-----------------+
|  count|             20016|            20016|             20016|             20016|               20016|            20016|
|   mean|2011.8071043165467|94.38114508393285| 5.391856864508392|1079.6401378896883| 4.852899898736011E7| 3.80398115507594|
| stddev| 17.98514699283888|36.40284037156159|2.7073543088835756| 2583.130487259596|1.1873400326246095E8|11.24234024848672|
|    min|              1902|                1|               0.0|                 0|                 1.0|              0.0|
|    max|              2026|              817|              10.0|             39245|       2.923706026E9|        1037.0135|
+-------

Dataset berisi 20.016 film dari 1902–2026, rata-rata durasi 94 menit dan rata-rata rating 5.39 dengan standar deviasi 2.71 yang besar, artinya sebaran kualitas film sangat beragam, mulai dari 0.0 hingga 10.0. Pendapatan rata-rata $48,5 juta namun standar deviasinya $118,7 juta, hanya segelintir film blockbuster yang mendominasi pendapatan, sementara mayoritas jauh di bawah rata-rata.

# Korelasi Numerik (Sederhana)

In [60]:
numeric_cols = ["TMDB Rating", "Votes", "Grossed in $", "Duration", "Popularity", "Year"]
 
for i, col1 in enumerate(numeric_cols):
    for col2 in numeric_cols[i+1:]:
        corr_val = movies_df.stat.corr(col1, col2)
        print(f"  Korelasi {col1:20} ↔ {col2:20} : {corr_val:+.4f}")

  Korelasi TMDB Rating          ↔ Votes                : +0.2468
  Korelasi TMDB Rating          ↔ Grossed in $         : +0.1307
  Korelasi TMDB Rating          ↔ Duration             : +0.4934
  Korelasi TMDB Rating          ↔ Popularity           : +0.1458
  Korelasi TMDB Rating          ↔ Year                 : -0.3305
  Korelasi Votes                ↔ Grossed in $         : +0.7198
  Korelasi Votes                ↔ Duration             : +0.2341
  Korelasi Votes                ↔ Popularity           : +0.1847
  Korelasi Votes                ↔ Year                 : -0.1220
  Korelasi Grossed in $         ↔ Duration             : +0.1749
  Korelasi Grossed in $         ↔ Popularity           : +0.2012
  Korelasi Grossed in $         ↔ Year                 : -0.0263
  Korelasi Duration             ↔ Popularity           : +0.1337
  Korelasi Duration             ↔ Year                 : -0.2028
  Korelasi Popularity           ↔ Year                 : -0.0461


Temuan paling mencolok dari analisis korelasi ini adalah hubungan yang sangat kuat antara Votes dan Grossed in $ dengan nilai korelasi +0.71. Ini masuk akal secara logis, film yang ditonton banyak orang di bioskop cenderung juga banyak diulas dan diberi penilaian, sehingga jumlah votes dan pendapatan box office bergerak searah secara signifikan. Ini adalah satu-satunya pasangan variabel yang memiliki hubungan kuat dalam dataset ini.


Sementara itu, TMDB Rating ternyata hanya memiliki hubungan yang lemah terhadap semua variabel lainnya. Artinya, sebuah film yang mendapat rating tinggi belum tentu menghasilkan pendapatan besar, ditonton banyak orang, atau menjadi viral. Kualitas film menurut pengguna TMDB tampaknya tidak terlalu menentukan kesuksesan komersialnya.


Yang menarik, korelasi antara Rating dan Tahun Rilis bernilai negatif. Ini mengindikasikan bahwa film-film lama cenderung mendapat rating lebih tinggi dibanding film baru.

# Analisis Film

## Distribusi Film per Dekade

In [61]:
movies_df.withColumn(
    "Decade", (F.col("Year") / 10).cast("int") * 10
).groupBy("Decade") \
 .agg(F.count("*").alias("Jumlah_Film")) \
 .orderBy("Decade") \
 .show()

+------+-----------+
|Decade|Jumlah_Film|
+------+-----------+
|  1900|          2|
|  1910|         15|
|  1920|         38|
|  1930|         72|
|  1940|        109|
|  1950|        208|
|  1960|        336|
|  1970|        544|
|  1980|        971|
|  1990|       1505|
|  2000|       2512|
|  2010|       4200|
|  2020|       9504|
+------+-----------+



## Golden Age of Cinema

In [62]:
movies_df.withColumn("Era",
    F.when(F.col("Year") < 1960, "Classic (<1960)")
     .when(F.col("Year") < 1980, "New Hollywood (1960–79)")
     .when(F.col("Year") < 2000, "Blockbuster Era (1980–99)")
     .when(F.col("Year") < 2015, "Digital Age (2000–14)")
     .otherwise("Modern (2015+)")
).groupBy("Era").agg(
    F.count("*").alias("Film Count"),
    F.round(F.avg("TMDB Rating"),    3).alias("Avg Rating"),
    F.round(F.avg("Grossed in $") / 1e6, 2).alias("Avg Revenue (Juta USD)"),
    F.round(F.avg("Votes"),          0).alias("Avg Votes"),
).orderBy("Era").show(truncate=30)

+-------------------------+----------+----------+----------------------+---------+
|                      Era|Film Count|Avg Rating|Avg Revenue (Juta USD)|Avg Votes|
+-------------------------+----------+----------+----------------------+---------+
|Blockbuster Era (1980–99)|      2476|     6.499|                 55.51|   1464.0|
|          Classic (<1960)|       444|      6.62|                 18.66|    851.0|
|    Digital Age (2000–14)|      4398|     6.481|                 77.13|   2162.0|
|           Modern (2015+)|     11818|      4.61|                 38.92|    616.0|
|  New Hollywood (1960–79)|       880|     6.711|                 30.11|    935.0|
+-------------------------+----------+----------+----------------------+---------+



## Film Hidden Gems

In [63]:
print("Film Rating Tinggi, Popularity Rendah, Revenue Rendah")
med_pop  = movies_df.approxQuantile("Popularity",    [0.5], 0.05)[0]
med_rev  = movies_df.approxQuantile("Grossed in $",[0.5], 0.05)[0]
med_vote = movies_df.approxQuantile("Votes", [0.5], 0.05)[0]

print(f"  Median Popularity      : {med_pop:.2f}")
print(f"  Median Revenue         : ${med_rev/1e6:.1f} Juta")
print()
 
movies_df.filter(
    (F.col("TMDB Rating") >= 8.0) &
    (F.col("Popularity")  <= med_pop) &
    (F.col("Grossed in $") <= med_rev) &
    (F.col("Votes") >= med_vote)
).select("Movie Name", "Year", "TMDB Rating", "Popularity", "Votes",
         F.round(F.col("Grossed in $")/1e6, 2).alias("Revenue (Juta USD)")) \
 .orderBy(F.desc("TMDB Rating")) \
 .show(truncate=35)

Film Rating Tinggi, Popularity Rendah, Revenue Rendah
  Median Popularity      : 2.58
  Median Revenue         : $21.8 Juta

+-----------------------------------+----+-----------+----------+-----+------------------+
|                         Movie Name|Year|TMDB Rating|Popularity|Votes|Revenue (Juta USD)|
+-----------------------------------+----+-----------+----------+-----+------------------+
|             Grave of the Fireflies|1988|      8.445|    0.0122| 6437|              0.84|
|                  Impossible Things|2021|      8.415|    2.5331|  420|             21.76|
|                 Dedicated to my ex|2019|        8.3|    1.9642|  513|              1.32|
|        La Leyenda de los Chaneques|2023|        8.3|    2.4477|  254|             21.76|
|    We All Loved Each Other So Much|1974|      8.289|    2.1459|  633|             21.76|
|                 Hotarubi no Mori e|2011|      8.251|    0.0086| 1204|             21.76|
|                             Baraka|1992|      8.223|  

Semua film hidden gem ini memiliki tingkat popularitas di bawah median 2.58 dan pendapatan di bawah median $21.8 juta, padahal kualitasnya bersaing dengan film-film blockbuster terkenal.

# Analisis Rating

In [64]:
print("  Distribusi Kategori Rating:")
movies_df.withColumn("Rating Category",
    F.when(F.col("TMDB Rating") >= 8.5, "Masterpiece (≥8.5)")
     .when(F.col("TMDB Rating") >= 7.5, "Excellent (7.5–8.4)")
     .when(F.col("TMDB Rating") >= 6.5, "Good (6.5–7.4)")
     .otherwise("Below Average (<6.5)")
).groupBy("Rating Category") \
 .agg(F.count("*").alias("Jumlah Film")) \
 .orderBy(F.desc("Jumlah Film")) \
 .show()

  Distribusi Kategori Rating:
+--------------------+-----------+
|     Rating Category|Jumlah Film|
+--------------------+-----------+
|Below Average (<6.5)|      10811|
|      Good (6.5–7.4)|       6788|
| Excellent (7.5–8.4)|       2026|
|  Masterpiece (≥8.5)|        391|
+--------------------+-----------+



## Top Film Berdasarkan Rating

In [65]:
med_votes = movies_df.approxQuantile("Votes", [0.5], 0.05)[0]

movies_df.select("Movie Name", "Year", "TMDB Rating", "Votes", "Director") \
    .filter(F.col("Votes") >= med_votes) \
    .orderBy(F.desc("TMDB Rating")) \
    .limit(10) \
    .show(truncate=40)

+---------------------------+----+-----------+-----+--------------------+
|                 Movie Name|Year|TMDB Rating|Votes|            Director|
+---------------------------+----+-----------+-----+--------------------+
|   The Shawshank Redemption|1994|      8.717|30023|      Frank Darabont|
|              The Godfather|1972|      8.687|22674|Francis Ford Coppola|
|      The Godfather Part II|1974|      8.572|13733|Francis Ford Coppola|
|           Schindler's List|1993|      8.567|17260|    Steven Spielberg|
|               12 Angry Men|1957|      8.557| 9853|        Sidney Lumet|
|      ¿Quieres ser mi hijo?|2023|      8.543|  301|       Ihtzi Hurtado|
| Selena Gomez: My Mind & Me|2022|      8.534|  582|     Alek Keshishian|
|              Spirited Away|2001|      8.533|18119|      Hayao Miyazaki|
|            The Dark Knight|2008|      8.528|35409|   Christopher Nolan|
|Dilwale Dulhania Le Jayenge|1995|      8.517| 4565|       Aditya Chopra|
+---------------------------+----+----

Daftar 10 besar film dengan rating tertinggi ini memperlihatkan beberapa pola yang menarik. The Shawshank Redemption (1994) menempati posisi puncak dengan rating 8.716 sekaligus memiliki votes terbanyak kedua sebesar 30.023. Posisi kedua dan ketiga ditempati oleh The Godfather dan The Godfather Part II, keduanya karya Francis Ford Coppola, menjadikannya satu-satunya sutradara yang memiliki dua film sekaligus dalam daftar 10 besar ini.

# Analisis Pendapatan (Box Office)

In [66]:
movies_df.agg(
    F.round(F.sum("Grossed in $") / 1e9, 3).alias("Total Pendapatan (Milyar $)"),
    F.round(F.avg("Grossed in $") / 1e6, 2).alias("Rata-rata (Juta $)"),
    F.round(F.max("Grossed in $") / 1e6, 2).alias("Pendapatan Tertinggi (Juta $)"),
    F.round(F.min("Grossed in $") / 1e6, 2).alias("Pendapatan Terendah (Juta $)"),
).show()

+---------------------------+------------------+-----------------------------+----------------------------+
|Total Pendapatan (Milyar $)|Rata-rata (Juta $)|Pendapatan Tertinggi (Juta $)|Pendapatan Terendah (Juta $)|
+---------------------------+------------------+-----------------------------+----------------------------+
|                    971.356|             48.53|                      2923.71|                         0.0|
+---------------------------+------------------+-----------------------------+----------------------------+



Total pendapatan seluruh film dalam dataset mencapai $746,4 miliar, angka yang sangat besar namun perlu dipahami konteksnya secara lebih dalam. Rata-rata pendapatan per film adalah $76,64 juta, namun seperti yang sudah terlihat dari analisis sebelumnya, angka rata-rata ini sangat menyesatkan karena distribusi pendapatan yang sangat tidak merata.


Kesenjangan antara film terlaris dan film dengan pendapatan terendah sangat ekstrem. Film terlaris berhasil meraup $2,92 miliar, hampir 38 kali lipat dari rata-rata, sementara ada film yang pendapatannya tercatat $0 merupakan film yang data revenuenya tidak tersedia dan diisi dengan nilai 0.

## Top 5 Film Terlaris

In [67]:
print("Top 5 Film Terlaris:")
movies_df.select(
    "Movie Name", "Year",
    F.round(F.col("Grossed in $") / 1e6, 2).alias("Pendapatan (Juta USD)")
).orderBy(F.desc("Grossed in $")) \
 .limit(5) \
 .show(truncate=40)

Top 5 Film Terlaris:
+------------------------+----+---------------------+
|              Movie Name|Year|Pendapatan (Juta USD)|
+------------------------+----+---------------------+
|                  Avatar|2009|              2923.71|
|       Avengers: Endgame|2019|              2799.44|
|Avatar: The Way of Water|2022|               2353.1|
|                 Titanic|1997|              2264.16|
|                Ne Zha 2|2025|              2259.82|
+------------------------+----+---------------------+



Avatar (2009) karya James Cameron masih bertahan di puncak dengan pendapatan $2,92 miliar, diikuti Avengers: Endgame (2019) di posisi kedua dengan $2,79 miliar. Keduanya adalah satu-satunya film yang menembus angka $2,5 miliar.
Menariknya, James Cameron hadir dua kali dalam daftar ini melalui Avatar dan Titanic (1997), menjadikannya sutradara paling dominan dalam hal pendapatan box office sepanjang sejarah. Sementara itu, Avatar: The Way of Water (2022) sebagai sekuel Avatar berhasil masuk posisi ketiga, membuktikan bahwa franchise Avatar tetap memiliki daya tarik komersial yang luar biasa meski dirilis 13 tahun setelah film pertamanya.


Yang paling mengejutkan adalah kehadiran Ne Zha 2 (2025) di posisi kelima dengan pendapatan $2,26 miliar. Ini adalah film animasi asal China yang membuktikan bahwa dominasi Hollywood di box office global mulai mendapat tantangan serius dari industri film Asia, khususnya China yang pasar domestiknya saja sudah mampu menghasilkan pendapatan setara film-film blockbuster Hollywood terbesar.

## Korelasi Fitur Numerik terhadap Revenue

In [68]:
print("Korelasi Semua Fitur Numerik terhadap Revenue:")
revenue_pairs = [
    ("TMDB Rating",  movies_df.stat.corr("TMDB Rating",  "Grossed in $")),
    ("Votes",        movies_df.stat.corr("Votes",         "Grossed in $")),
    ("Duration",     movies_df.stat.corr("Duration",      "Grossed in $")),
    ("Popularity",   movies_df.stat.corr("Popularity",    "Grossed in $"))]
for feat, r in sorted(revenue_pairs, key=lambda x: -abs(x[1])):
    bar = "█" * int(abs(r) * 20)
    sign = "+" if r > 0 else "-"
    print(f"{feat:<18} {sign}{abs(r):.4f}  {bar}")

Korelasi Semua Fitur Numerik terhadap Revenue:
Votes              +0.7198  ██████████████
Popularity         +0.2012  ████
Duration           +0.1749  ███
TMDB Rating        +0.1307  ██


Dari seluruh fitur numerik yang dianalisis, Votes adalah prediktor terkuat terhadap revenue dengan korelasi +0.71 yang tergolong kuat. Ini sangat masuk akal, film yang banyak ditonton orang di bioskop akan menghasilkan pendapatan besar sekaligus mendorong lebih banyak orang untuk memberikan penilaian, sehingga kedua variabel ini saling memperkuat satu sama lain. Dengan kata lain, votes bisa dijadikan indikator awal yang cukup andal untuk memperkirakan seberapa besar pendapatan sebuah film.


Sementara itu, Popularity, Duration, dan TMDB Rating hanya memiliki korelasi lemah terhadap revenue. Artinya, seberapa viral sebuah film di platform TMDB, seberapa panjang durasinya, maupun seberapa tinggi ratingnya tidak terlalu menentukan apakah film tersebut akan menghasilkan uang banyak atau tidak.

## Rata-rata Revenue per Kategori Rating

In [69]:
med_votes = movies_df.approxQuantile("Votes", [0.5], 0.05)[0]

movies_df.filter(F.col("Votes") >= med_votes) \
.withColumn("Rating Tier",
    F.when(F.col("TMDB Rating") >= 8.5, "S-tier (≥8.5)")
     .when(F.col("TMDB Rating") >= 8.0, "A-tier (8.0–8.4)")
     .when(F.col("TMDB Rating") >= 7.0, "B-tier (7.0–7.9)")
     .otherwise("C-tier (<7.0)")
).groupBy("Rating Tier").agg(
    F.count("*").alias("Film Count"),
    F.round(F.avg("Grossed in $") / 1e6, 2).alias("Avg Revenue (Juta USD)"),
    F.round(F.sum("Grossed in $") / 1e6, 2).alias("Total Revenue (Juta USD)")
).orderBy(F.desc("Avg Revenue (Juta USD)")) \
.show()

+----------------+----------+----------------------+------------------------+
|     Rating Tier|Film Count|Avg Revenue (Juta USD)|Total Revenue (Juta USD)|
+----------------+----------+----------------------+------------------------+
|   S-tier (≥8.5)|        14|                210.26|                 2943.62|
|A-tier (8.0–8.4)|       291|                143.27|                41692.24|
|B-tier (7.0–7.9)|      3334|                 87.57|               291956.91|
|   C-tier (<7.0)|      6759|                 62.95|               425452.32|
+----------------+----------+----------------------+------------------------+



Kualitas film berbanding lurus dengan rata-rata pendapatannya. Semakin tinggi rating tier sebuah film, semakin besar pula rata-rata pendapatan yang diperoleh.

# Analisis Genre

In [70]:
# Explode genre ke baris terpisah
genre_df = movies_df.withColumn(
    "Genre_Split",
    F.explode(F.split(F.col("Genre"), ",\\s*"))
)

In [71]:
print("  Frekuensi Genre:")
genre_df.groupBy("Genre_Split") \
        .agg(
            F.count("*").alias("Jumlah Film"),
            F.round(F.avg("TMDB Rating"), 3).alias("Avg Rating"),
            F.round(F.avg("Grossed in $") / 1e6, 2).alias("Avg Pendapatan (Juta USD)")
        ) \
        .orderBy(F.desc("Jumlah Film")) \
        .show()

  Frekuensi Genre:
+---------------+-----------+----------+-------------------------+
|    Genre_Split|Jumlah Film|Avg Rating|Avg Pendapatan (Juta USD)|
+---------------+-----------+----------+-------------------------+
|          Drama|       8600|      5.84|                    35.53|
|         Comedy|       5951|     5.834|                    51.54|
|       Thriller|       4381|     5.868|                    45.87|
|         Action|       3815|     6.241|                    89.52|
|        Romance|       2928|     6.056|                    43.19|
|         Horror|       2651|     5.197|                    31.21|
|          Crime|       2516|     6.232|                    44.22|
|      Adventure|       2346|     6.448|                   139.38|
|        Fantasy|       1884|     6.217|                    85.34|
|      Animation|       1878|     6.017|                    71.81|
|Science Fiction|       1795|     6.045|                    99.17|
|         Family|       1646|     6.241|   

Drama tetap paling banyak diproduksi dengan rating tertinggi, sementara Adventure dan Sci-Fi paling menguntungkan secara komersial. Rating terendah merupakan genre dokumenter.

## Window Function - Ranking per Genre

In [72]:
genre_decade = movies_df.withColumn(
    "Decade", (F.col("Year") / 10).cast("int") * 10
).withColumn(
    "Genre Single", F.explode(F.split(F.col("Genre"), ",\\s*"))
)

In [73]:
from pyspark.sql.window import Window

result = genre_decade.groupBy("Decade", "Genre Single").agg(
    F.count("*").alias("Film Count"),
    F.round(F.avg("TMDB Rating"), 3).alias("Avg Rating"),
)

windowSpec = Window.partitionBy("Decade").orderBy(F.desc("Film Count"))

result.withColumn("rank", F.row_number().over(windowSpec)) \
      .filter(F.col("rank") <= 2) \
      .orderBy("Decade", "rank") \
      .show(truncate=20)

+------+------------+----------+----------+----+
|Decade|Genre Single|Film Count|Avg Rating|rank|
+------+------------+----------+----------+----+
|  1900|   Adventure|         2|     7.448|   1|
|  1900|     Fantasy|         1|     7.905|   2|
|  1910|       Drama|         9|     2.667|   1|
|  1910|      Comedy|         5|     4.104|   2|
|  1920|      Comedy|        20|      6.56|   1|
|  1920|       Drama|        19|      5.58|   2|
|  1930|       Drama|        33|     6.504|   1|
|  1930|      Comedy|        22|     6.838|   2|
|  1940|       Drama|        60|     7.304|   1|
|  1940|     Romance|        31|     7.407|   2|
|  1950|       Drama|       120|     7.168|   1|
|  1950|     Romance|        58|     7.006|   2|
|  1960|       Drama|       183|     7.173|   1|
|  1960|      Comedy|        89|     7.018|   2|
|  1970|       Drama|       257|      6.72|   1|
|  1970|      Comedy|       170|     6.521|   2|
|  1980|      Comedy|       375|     6.563|   1|
|  1980|       Drama

# Analisis Sutradara

## Sutradara dengan Film paling Stabil Kualitasnya

In [74]:
director_stats = movies_df.groupBy("Director").agg(
    F.count("*").alias("Film Count"),
    F.round(F.avg("TMDB Rating"),    3).alias("Avg Rating"),
    F.round(F.stddev("TMDB Rating"), 3).alias("Stddev Rating"),
    F.round(F.avg("Grossed in $") / 1e6, 2).alias("Avg Revenue (Juta USD)"),
    F.round(F.sum("Grossed in $") / 1e6, 2).alias("Total Revenue (Juta USD)"),
)
director_stats.orderBy(F.desc("Film Count")).show(10, truncate=25)

+-----------------+----------+----------+-------------+----------------------+------------------------+
|         Director|Film Count|Avg Rating|Stddev Rating|Avg Revenue (Juta USD)|Total Revenue (Juta USD)|
+-----------------+----------+----------+-------------+----------------------+------------------------+
|      Woody Allen|        45|     6.779|        0.488|                 29.06|                 1307.58|
|   Clint Eastwood|        36|     6.908|        0.613|                 96.14|                 3460.87|
| Steven Spielberg|        34|     7.209|        0.629|                312.86|                10637.39|
| Alfred Hitchcock|        31|     7.329|        0.558|                  12.1|                   375.0|
|     Ridley Scott|        28|     6.839|        0.705|                178.21|                  4989.8|
|Steven Soderbergh|        26|     6.247|        1.348|                 95.12|                 2473.01|
|  Martin Scorsese|        26|     7.412|        0.534|         

Woody Allen paling produktif (45 film) tapi revenue kecil ($29 juta per film). Steven Spielberg paling dominan, rata-rata rating 7.21, revenue $312,86 juta per film, dan total revenue $10,6 miliar. Akira Kurosawa memiliki rata-rata rating tertinggi (7.68) meski revenue kecil karena era dan pasar berbeda.

# Analisis Aktor

In [75]:
med_votes = movies_df.approxQuantile("Votes", [0.5], 0.05)[0]
filtered_movies = movies_df.filter(F.col("Votes") >= med_votes)

In [76]:
# Ambil Top 50 Aktor
top_50 = filtered_movies.withColumn("Actor", F.explode("Stars")) \
    .groupBy("Actor").count() \
    .orderBy(F.desc("count")) \
    .limit(50).collect()

top_50_actors = [row["Actor"] for row in top_50]

In [77]:
# Buat df khusus aktor
actor_df = filtered_movies.withColumn("Actor", F.explode("Stars")) \
    .filter(F.col("Actor").isin(top_50_actors))

In [78]:
# Rata-rata Rating per Aktor
actor_rating = actor_df.groupBy("Actor").agg(
    F.count("*").alias("Film Count"),
    F.round(F.avg("TMDB Rating"), 3).alias("Avg Rating"),
    F.round(F.min("TMDB Rating"), 3).alias("Min Rating"),
    F.round(F.max("TMDB Rating"), 3).alias("Max Rating"),
    F.round(F.avg("Grossed in $") / 1e6, 2).alias("Avg Revenue (Juta USD)")
)

## Aktor dengan Rating di Atas Rata-Rata Global

In [79]:
global_avg = filtered_movies.agg(F.avg("TMDB Rating")).first()[0]
print(f"Aktor dengan Avg Rating di atas Global ({global_avg:.3f}) (Votes ≥ Median: {med_votes:.0f})")

actor_rating.filter(
    (F.col("Film Count") >= 2) &
    (F.col("Avg Rating") > global_avg)
).orderBy(F.desc("Avg Rating")) \
 .show(10, truncate=30)

Aktor dengan Avg Rating di atas Global (6.695) (Votes ≥ Median: 248)
+-----------------+----------+----------+----------+----------+----------------------+
|            Actor|Film Count|Avg Rating|Min Rating|Max Rating|Avg Revenue (Juta USD)|
+-----------------+----------+----------+----------+----------+----------------------+
|        Brad Pitt|        41|     7.009|     5.516|     8.438|                195.52|
|        Tom Hanks|        57|     6.951|     5.524|     8.504|                199.66|
|    Ralph Fiennes|        37|     6.934|     5.758|     8.567|                176.52|
|   Clint Eastwood|        43|      6.91|     5.763|     8.468|                  60.1|
|       Tom Cruise|        39|     6.907|     5.517|     8.152|                332.42|
|   Dustin Hoffman|        38|     6.879|     5.654|       7.8|                 151.8|
|        Al Pacino|        40|     6.861|     4.392|     8.687|                 72.03|
|       Matt Damon|        45|     6.848|     5.399|     8.15

## Aktor dengan Film Terbanyak

In [80]:
actor_rating.orderBy(F.desc("Film Count")).show(10, truncate=30)

+-----------------+----------+----------+----------+----------+----------------------+
|            Actor|Film Count|Avg Rating|Min Rating|Max Rating|Avg Revenue (Juta USD)|
+-----------------+----------+----------+----------+----------+----------------------+
|   Robert De Niro|        72|     6.831|     5.533|     8.572|                 95.53|
|     Nicolas Cage|        67|     6.248|       4.3|     7.449|                 82.58|
|Samuel L. Jackson|        63|     6.496|       4.7|     8.486|                147.75|
|     Bruce Willis|        62|     6.284|     3.986|     8.486|                115.03|
|        Tom Hanks|        57|     6.951|     5.524|     8.504|                199.66|
|    Nicole Kidman|        51|     6.407|     5.083|       8.0|                 75.07|
|    Mark Wahlberg|        51|     6.578|     5.394|     8.159|                138.12|
|   Morgan Freeman|        50|     6.568|     4.723|     8.717|                116.61|
|      Jackie Chan|        50|     6.665|  

## Pengaruh Aktor terhadap Revenue

In [81]:
actor_rating.orderBy(F.desc("Avg Revenue (Juta USD)")).show(10, truncate=30)

+--------------+----------+----------+----------+----------+----------------------+
|         Actor|Film Count|Avg Rating|Min Rating|Max Rating|Avg Revenue (Juta USD)|
+--------------+----------+----------+----------+----------+----------------------+
|    Tom Cruise|        39|     6.907|     5.517|     8.152|                332.42|
|   Johnny Depp|        44|      6.78|       5.6|     7.821|                218.95|
| Harrison Ford|        39|     6.747|     5.558|     8.392|                213.08|
|     Tom Hanks|        57|     6.951|     5.524|     8.504|                199.66|
|     Brad Pitt|        41|     7.009|     5.516|     8.438|                195.52|
| Ralph Fiennes|        37|     6.934|     5.758|     8.567|                176.52|
| Ryan Reynolds|        40|     6.577|      5.18|     7.622|                174.36|
|    Matt Damon|        45|     6.848|     5.399|     8.159|                168.77|
|  Eddie Murphy|        42|     6.333|       5.3|     7.901|                

Robert De Niro paling banyak film (75), tapi Tom Hanks paling seimbang antara kualitas (6.97) dan revenue ($196 juta). Tom Cruise tetap raja box office dengan rata-rata $324 juta per film. Brad Pitt terbaik kombinasi rating (7.01) dan revenue ($195 juta).

## Pasangan Aktor Paling Sering Tampil Bersama

In [82]:
# Self-join actor_df untuk menemukan pasangan
a1 = actor_df.select(
    F.col("Movie Name"), F.col("Actor").alias("Actor A"), F.col("TMDB Rating")
)
a2 = actor_df.select(
    F.col("Movie Name"), F.col("Actor").alias("Actor B")
)
 
pair_df = a1.join(a2, on="Movie Name") \
            .filter(F.col("Actor A") < F.col("Actor B"))  # hindari duplikasi
 
print("Top 10 Pasangan Aktor (Film Bersama Terbanyak):")
pair_df.groupBy("Actor A", "Actor B").agg(
    F.count("*").alias("Films Together"),
    F.round(F.avg("TMDB Rating"), 3).alias("Avg Rating Together"),
).orderBy(F.desc("Films Together"), F.desc("Avg Rating Together")) \
 .show(10, truncate=28)

Top 10 Pasangan Aktor (Film Bersama Terbanyak):
+----------------+-----------------+--------------+-------------------+
|         Actor A|          Actor B|Films Together|Avg Rating Together|
+----------------+-----------------+--------------+-------------------+
|     Ben Affleck|       Matt Damon|             5|              7.368|
|       Al Pacino|   Robert De Niro|             4|               7.49|
|    Bruce Willis|Samuel L. Jackson|             4|              7.401|
|  Dustin Hoffman|   Robert De Niro|             4|              6.611|
|    Ben Kingsley|    Ralph Fiennes|             3|              7.558|
|       Brad Pitt|       Matt Damon|             3|              6.915|
|   Nicole Kidman|       Tom Cruise|             3|              6.867|
|   Kevin Costner|  Tommy Lee Jones|             3|              6.778|
|   Ryan Reynolds|Samuel L. Jackson|             3|              6.605|
|Antonio Banderas|     Eddie Murphy|             3|              6.353|
+---------------

## Kolaborasi Director dan Actor

In [83]:
collab_df = actor_df.groupBy("Director", "Actor").agg(
    F.count("*").alias("Films Together"),
    F.round(F.avg("TMDB Rating"),    3).alias("Avg Rating"),
    F.round(F.avg("Grossed in $") / 1e6, 2).alias("Avg Revenue (Juta USD)"),
)
 
print("Pasangan Direktur–Aktor Paling Sering Berkolaborasi:")
collab_df.filter(F.col("Films Together") >= 2) \
         .orderBy(F.desc("Films Together"), F.desc("Avg Rating")) \
         .show(10, truncate=30)

Pasangan Direktur–Aktor Paling Sering Berkolaborasi:
+---------------------+------------------+--------------+----------+----------------------+
|             Director|             Actor|Films Together|Avg Rating|Avg Revenue (Juta USD)|
+---------------------+------------------+--------------+----------+----------------------+
|       Clint Eastwood|    Clint Eastwood|            22|     6.844|                 82.24|
|      Martin Scorsese|    Robert De Niro|            10|     7.611|                 57.91|
|          Jackie Chan|       Jackie Chan|             8|     7.081|                 37.52|
|         Dennis Dugan|      Adam Sandler|             8|     6.075|                193.85|
|           Tim Burton|       Johnny Depp|             7|     7.048|                313.91|
|      Pedro Almodóvar|  Antonio Banderas|             6|     6.978|                 17.06|
|       Richard Donner|        Mel Gibson|             6|     6.855|                212.54|
|   Sylvester Stallone|Sylv

# Analisis Durasi

In [85]:
print("Kategori Durasi:")
movies_df.withColumn("Durasi Kategori",
    F.when(F.col("Duration") < 90,  "Pendek (<90 mnt)")
     .when(F.col("Duration") < 120, "Standar (90–119 mnt)")
     .when(F.col("Duration") < 150, "Panjang (120–149 mnt)")
     .otherwise("Epic (≥150 mnt)")
).groupBy("Durasi Kategori") \
 .agg(
     F.count("*").alias("Jumlah Film"),
     F.round(F.avg("TMDB Rating"), 3).alias("Avg Rating")
 ) \
 .orderBy("Durasi Kategori") \
 .show()

Kategori Durasi:
+--------------------+-----------+----------+
|     Durasi Kategori|Jumlah Film|Avg Rating|
+--------------------+-----------+----------+
|     Epic (≥150 mnt)|        606|     6.419|
|Panjang (120–149 ...|       2836|     6.537|
|    Pendek (<90 mnt)|       5919|       3.7|
|Standar (90–119 mnt)|      10655|     5.969|
+--------------------+-----------+----------+



Film Pendek (<90 mnt) rating rendah (3.7), banyak diisi film berkualitas rendah. Film Panjang rating tertinggi (6.53).

# Analisis Sertifikasi

In [86]:
movies_df.groupBy("Certificate") \
         .agg(
             F.count("*").alias("Jumlah Film"),
             F.round(F.avg("TMDB Rating"), 3).alias("Avg Rating")
         ) \
         .orderBy(F.desc("Jumlah Film")) \
         .show()

+-----------+-----------+----------+
|Certificate|Jumlah Film|Avg Rating|
+-----------+-----------+----------+
|    Unknown|       8892|     4.092|
|          R|       4815|     6.443|
|      PG-13|       2699|     6.513|
|         PG|       1605|     6.623|
|         NR|       1404|     5.914|
|          G|        525|     6.759|
|      NC-17|         76|     5.964|
+-----------+-----------+----------+



Film G rating tertinggi (6.76) didorong film animasi berkualitas. Film R terbanyak di antara yang bersertifikasi (4.815 film).